# 🇮🇩 Modul 03: Segmentasi Komoditas Pangan Menggunakan K-Means Clustering
### Indonesian Food Commodity Price Intelligence Platform — Machine Learning Segmentation
**Author**: Fauzi Noorsyabani · Full Stack AI Engineer  
**Dataset**: Harga Pangan Indonesia (2023–2025)

---
### 🎯 Tujuan Analisis
1. Melakukan rekayasa fitur multi-dimensi: `mean_price` (level harga), `cv_pct` (volatilitas), dan `total_growth` (laju inflasi kumulatif).
2. Menerapkan standardisasi `StandardScaler` untuk mencegah bias skala nominal rupiah.
3. Menentukan jumlah klaster optimal via **Silhouette Score Analysis** (menguji k=2,3,4,5).
4. Melakukan reduksi dimensi PCA 2D untuk memproyeksikan lanskap pengelompokan pasar pangan.

In [ ]:
# Environment Setup & Library Installation
# Cell ini kompatibel untuk dijalankan di Google Colab maupun Jupyter Notebook lokal

import sys
import subprocess

# Instalasi paket jika belum tersedia (terutama di Google Colab)
required_packages = ['pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'beautifulsoup4', 'lxml']
for pkg in required_packages:
    try:
        __import__(pkg)
    except ImportError:
        print(f"📦 Menginstal {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import os
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from bs4 import BeautifulSoup

# Pengaturan styling visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.edgecolor'] = '#CCCCCC'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['grid.alpha'] = 0.7

print("✅ Lingkungan analisis siap digunakan (Local & Google Colab compatible)")


In [ ]:
# Fungsi Universal Pemuatan Data (Local & Colab Compatible)
def load_harga_pangan():
    # Cek lokasi file di berbagai kemungkinan path
    possible_paths = [
        "harga-rata-rata-bahan-pangan-komponen-data.xls",
        "../harga-rata-rata-bahan-pangan-komponen-data.xls",
        "data/harga_pangan_clean.csv",
        "../data/harga_pangan_clean.csv",
        "/content/harga-rata-rata-bahan-pangan-komponen-data.xls"
    ]
    
    file_path = None
    for p in possible_paths:
        if os.path.exists(p):
            file_path = p
            break
            
    if file_path is None:
        # Fallback embedded dataset jika file xls belum diunggah ke Colab
        print("⚠️ File lokal tidak ditemukan. Menggunakan embedded dataset resmi BPS/Kemendag (33 data point)...")
        raw_html = """
        <table>
          <thead>
            <tr><th><b>periode_data</b></th><th><b>komoditas</b></th><th><b>harga_rata_rata</b></th></tr>
          </thead>
          <tbody>
            <tr><td>2023</td><td>BERAS IR I</td><td>12,759.81</td></tr>
            <tr><td>2023</td><td>BERAS IR III</td><td>11,238.67</td></tr>
            <tr><td>2023</td><td>CABAI RAWIT MERHA</td><td>62,417.22</td></tr>
            <tr><td>2023</td><td>CABAI MERAH KERITING</td><td>51,684.27</td></tr>
            <tr><td>2023</td><td>BAWANG MERAH</td><td>39,974.05</td></tr>
            <tr><td>2023</td><td>BAWANG PUTIH</td><td>37,997.28</td></tr>
            <tr><td>2023</td><td>DAGING AYAM</td><td>40,142.56</td></tr>
            <tr><td>2023</td><td>TELUR AYAM</td><td>29,005.93</td></tr>
            <tr><td>2023</td><td>DAGING SAPI</td><td>140,288.21</td></tr>
            <tr><td>2023</td><td>GULA PASIR</td><td>14,954.54</td></tr>
            <tr><td>2023</td><td>MINYAK GORENG</td><td>15,777.51</td></tr>
            <tr><td>2025</td><td>BERAS IR I</td><td>15169</td></tr>
            <tr><td>2025</td><td>BERAS IR III</td><td>13599</td></tr>
            <tr><td>2025</td><td>CABAI RAWIT MERAH</td><td>70942</td></tr>
            <tr><td>2025</td><td>CABAI MERAH KERITING</td><td>59892</td></tr>
            <tr><td>2025</td><td>BAWANG MERAH</td><td>50531</td></tr>
            <tr><td>2025</td><td>BAWANG PUTIH</td><td>44513</td></tr>
            <tr><td>2025</td><td>DAGING AYAM</td><td>41071</td></tr>
            <tr><td>2025</td><td>TELUR AYAM</td><td>29297</td></tr>
            <tr><td>2025</td><td>DAGING SAPI</td><td>138353</td></tr>
            <tr><td>2025</td><td>GULA PASIR</td><td>18430</td></tr>
            <tr><td>2025</td><td>MINYAK GORENG</td><td>19652</td></tr>
            <tr><td>2024</td><td>BERAS IR I</td><td>14734</td></tr>
            <tr><td>2024</td><td>BERAS IR III</td><td>13517</td></tr>
            <tr><td>2024</td><td>CABAI RAWIT MERHA</td><td>59735</td></tr>
            <tr><td>2024</td><td>CABAI MERAH KERITING</td><td>53865</td></tr>
            <tr><td>2024</td><td>BAWANG MERAH</td><td>41780</td></tr>
            <tr><td>2024</td><td>BAWANG PUTIH</td><td>44106</td></tr>
            <tr><td>2024</td><td>DAGING AYAM</td><td>40525</td></tr>
            <tr><td>2024</td><td>TELUR AYAM</td><td>28631</td></tr>
            <tr><td>2024</td><td>DAGING SAPI</td><td>138588</td></tr>
            <tr><td>2024</td><td>GULA PASIR</td><td>17434</td></tr>
            <tr><td>2024</td><td>MINYAK GORENG</td><td>17033</td></tr>
          </tbody>
        </table>
        """
        soup = BeautifulSoup(raw_html, "html.parser")
    else:
        print(f"📖 Membaca berkas data dari: {file_path}")
        if file_path.endswith('.csv'):
            return pd.read_csv(file_path)
        with open(file_path, "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f.read(), "html.parser")

    table = soup.find("table")
    rows = []
    for tr in table.find_all("tr"):
        cells = [c.get_text(strip=True) for c in tr.find_all(["td", "th"])]
        if cells:
            rows.append(cells)
            
    df = pd.DataFrame(rows[1:], columns=[c.strip().lower() for c in rows[0]])
    df["periode_data"] = df["periode_data"].astype(int)
    df["harga_rata_rata"] = df["harga_rata_rata"].astype(str).str.replace(",", "", regex=False).astype(float)
    df["komoditas"] = df["komoditas"].str.strip().str.upper().replace({"CABAI RAWIT MERHA": "CABAI RAWIT MERAH"})
    df = df.sort_values(["periode_data", "komoditas"]).reset_index(drop=True)
    return df

df = load_harga_pangan()
display(df.head(11))


## 1. Rekayasa Fitur & Evaluasi Silhouette Score

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

pivot = df.pivot_table(index='komoditas', columns='periode_data', values='harga_rata_rata')
pivot.columns = [int(c) for c in pivot.columns]

# Rekayasa Fitur
features = pd.DataFrame(index=pivot.index)
features['mean_price'] = pivot.mean(axis=1)
features['std_price'] = pivot.std(axis=1)
features['cv_pct'] = (features['std_price'] / features['mean_price']) * 100
features['total_growth_pct'] = ((pivot[2025] - pivot[2023]) / pivot[2023]) * 100

# Standarisasi
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features[['mean_price', 'cv_pct', 'total_growth_pct']])

# Eksplorasi Silhouette Score
sil_scores = {}
for k in range(2, 6):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil_scores[k] = silhouette_score(X_scaled, labels)
    print(f"K = {k} -> Silhouette Score = {sil_scores[k]:.4f}")

best_k = max(sil_scores, key=sil_scores.get)
print(f"\n🎯 Optimal Cluster K: {best_k} (Silhouette = {sil_scores[best_k]:.4f})")

### 🔍 Analisis Silhouette
Jumlah klaster optimal adalah **K = 3** dengan skor silhouette tertinggi **0.6038** (skor > 0.50 menandakan struktur pemisahan klaster yang kuat dan padat).

In [ ]:
# Pemodelan K-Means Final & Visualisasi PCA 2D
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
features['cluster'] = km_final.fit_predict(X_scaled)

# Reduksi Dimensi 2D via PCA
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)
features['pca_1'] = coords[:, 0]
features['pca_2'] = coords[:, 1]

# Pemetaan Label Bisnis
cluster_means = features.groupby('cluster')['mean_price'].mean().sort_values()
tier_map = {c_id: f"Tier {name}" for c_id, name in zip(cluster_means.index, ["Terjangkau & Volatil", "Stabil Terkendali", "Premium Ternak"])}
features['segmen'] = features['cluster'].map(tier_map)

fig, ax = plt.subplots(figsize=(12, 7))
sns.scatterplot(
    data=features,
    x='pca_1',
    y='pca_2',
    hue='segmen',
    s=features['mean_price'] / 180,
    palette='Set2',
    edgecolor='black',
    linewidth=1.2,
    ax=ax
)

for idx, row in features.iterrows():
    ax.annotate(idx, (row['pca_1'], row['pca_2']), xytext=(6, 6), textcoords='offset points', fontweight='bold', fontsize=9)

ax.set_title(f'Proyeksi Segmentasi Komoditas Pangan via PCA 2D (Silhouette = {sil_scores[best_k]:.4f})', fontsize=13, fontweight='bold')
ax.set_xlabel(f'PCA 1 ({pca.explained_variance_ratio_[0]*100:.1f}% Varians)', fontweight='bold')
ax.set_ylabel(f'PCA 2 ({pca.explained_variance_ratio_[1]*100:.1f}% Varians)', fontweight='bold')
plt.legend(title='Segmen Pasar', loc='upper right')
plt.tight_layout()
plt.show()

# Ringkasan Karakteristik Segmen
display(features.groupby('segmen')[['mean_price', 'cv_pct', 'total_growth_pct']].mean().round(2))

---
### Final Summary
### Q&A
- **Q**: Berapa jumlah klaster komoditas optimal dan apa interpretasinya?  
  **A**: 3 Klaster: Tier Terjangkau & Volatil (sembako & hortikultura), Tier Stabil Terkendali (Daging Ayam & Telur), dan Tier Premium (Daging Sapi).

### Data Analysis Key Findings
- K-Means berhasil memisahkan komoditas dengan akurasi pemisahan tinggi (Silhouette Score = **0.6038**).
- Komoditas protein unggas (Ayam & Telur) memiliki stabilitas harga tertinggi dengan CV hanya 1.15%.

### Insights or Next Steps
- Klaster Terjangkau & Volatil memerlukan mitigasi rantai pasok dinamis karena volatilitas harganya tinggi (>10%).